# Notebook 07 - Detect and extract objects

*Time: about 55-60 minutes*

## What you will learn
- **HSV colour space** and `cv2.inRange`: pick out everything of one colour.
- `cv2.bitwise_and` with a **mask**: keep only the masked colour.
- Find objects with contours, **count** them and **crop** each one into its own picture (and save it).
- Filter a messy scene by **area**, **aspect ratio** and **circularity**.
- Find one *known* object with `cv2.matchTemplate`, and avoid duplicate detections (**non-max suppression**).

## Prerequisites
- Notebooks 03 and 04 (thresholding, contours, boxes).

### What this code does

We load the libraries. `os` creates folders and checks files.

In [ ]:
import sys                                    # lets us change where Python looks for files
sys.path.append("..")                         # also look one folder up (utils.py lives there)
import os                                     # folders and files
import cv2                                    # OpenCV
import numpy as np                            # NumPy: colour ranges and maths
from utils import load, show, show_many       # our helpers
print("Libraries loaded - ready to go!")      # a visible sign that everything worked

### What you should see / Why we care

`Libraries loaded - ready to go!`

## Part 1: finding objects by colour

### What this code does

`IMAGE_PATH` is the picture of coloured shapes (**change this line for another picture**). We load it and look at it.

In [ ]:
IMAGE_PATH = "../assets/objects_shapes.png"   # <- change me
SCENE_PATH = "../assets/objects_scene.png"    # the messy scene for Part 2
img = load(IMAGE_PATH)                        # colour picture (BGR)
print("shape:", img.shape)                    # height, width, channels
show(img, "red circles, green squares, blue triangles", size=(7, 5))

### What you should see / Why we care

`shape: (420, 600, 3)` and the picture: 3 red circles, 3 green squares, 3 blue triangles on white.

### What this code does

Colours in BGR are awkward ("how much blue makes it blue-ish?"). **HSV** separates colour into: **H**ue (the colour itself, 0-179 in OpenCV), **S**aturation (how vivid) and **V**alue (how bright). In HSV "find green" simply means "hue near 60". We convert and print the HSV of three pixels.

In [ ]:
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)    # convert to Hue, Saturation, Value
print("red circle pixel   (H, S, V):", hsv[90, 100])     # hue near 0
print("green square pixel (H, S, V):", hsv[210, 105])    # hue near 60
print("blue triangle pixel(H, S, V):", hsv[350, 110])    # hue near 120
print("white background   (H, S, V):", hsv[5, 5])        # saturation 0 = no colour

### What you should see / Why we care

Hue is **0** for red, **60** for green and **112** for our blue (pure blue would be 120). The white background has saturation **0**. That is why we can ignore it with a minimum saturation.

### What this code does

`cv2.inRange(hsv, lower, upper)` returns a **mask**: white (255) where the pixel is inside the colour range, black (0) elsewhere. Red is special: its hue sits at both ends of the scale (0-10 and 170-179), so we make **two** masks and combine them with `cv2.bitwise_or`. Saturation and value must be at least 100, so white and gray are ignored.

In [ ]:
low_red1, high_red1 = np.array([0, 100, 100]), np.array([10, 255, 255])      # red, low end
low_red2, high_red2 = np.array([170, 100, 100]), np.array([179, 255, 255])   # red, high end
red_mask = cv2.bitwise_or(cv2.inRange(hsv, low_red1, high_red1),
                          cv2.inRange(hsv, low_red2, high_red2))             # either end
green_mask = cv2.inRange(hsv, np.array([40, 100, 100]), np.array([80, 255, 255]))
blue_mask = cv2.inRange(hsv, np.array([100, 100, 100]), np.array([130, 255, 255]))
show_many([red_mask, green_mask, blue_mask], ["red mask", "green mask", "blue mask"],
          columns=3, size=(12, 3))

### What you should see / Why we care

Three black pictures, each with **white shapes of only one colour**: circles, squares, triangles. We have separated the objects by colour.

### What this code does

`cv2.bitwise_and(img, img, mask=mask)` keeps the original pixels only where the mask is white, and paints the rest black.

In [ ]:
only_red = cv2.bitwise_and(img, img, mask=red_mask)        # keep only red things
only_green = cv2.bitwise_and(img, img, mask=green_mask)    # keep only green things
only_blue = cv2.bitwise_and(img, img, mask=blue_mask)      # keep only blue things
show_many([only_red, only_green, only_blue], ["only red", "only green", "only blue"],
          columns=3, size=(12, 3))

### What you should see / Why we care

Each picture shows the objects of one colour *in their real colour* on a black background.

### What this code does

We store the three masks in a dictionary (a labelled drawer) and, for each colour, find the contours (white shapes on black: the masks already follow the golden rule!), keep those with an area of at least 200 and count them. We also draw a magenta box around each.

In [ ]:
masks = {"red": red_mask, "green": green_mask, "blue": blue_mask}   # name -> mask
boxed = img.copy()                                    # picture to draw on
objects = []                                          # every box found, with its colour name
for name in masks:                                    # red, then green, then blue
    contours, _ = cv2.findContours(masks[name], cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    count = 0                                         # objects of this colour
    for contour in contours:
        if cv2.contourArea(contour) >= 200:           # ignore tiny leftovers
            x, y, w, h = cv2.boundingRect(contour)
            cv2.rectangle(boxed, (x, y), (x + w, y + h), (255, 0, 255), 2)
            objects.append((name, x, y, w, h))
            count = count + 1
    print(name, "objects:", count)
show(boxed, "every object found", size=(7, 5))

### What you should see / Why we care

`red objects: 3`, `green objects: 3`, `blue objects: 3`, and a magenta box around all 9 shapes.

### What this code does

**Extraction.** For every object we slice its box out of the original picture, give it a title (its colour) and show all crops in a grid. Then we **save one crop to a file** with `cv2.imwrite` and read it back to prove it worked.

In [ ]:
crops, titles = [], []                                # small pictures and their titles
for name, x, y, w, h in objects:                      # every object found
    crops.append(img[y:y + h, x:x + w])               # cut it out of the colour picture
    titles.append(name + " " + str(w) + "x" + str(h)) # e.g. "red 82x82"
show_many(crops, titles, columns=5, size=(14, 5))
os.makedirs("../assets/out", exist_ok=True)           # make sure the folder exists
cv2.imwrite("../assets/out/first_object.png", crops[0])   # save the first crop
saved = cv2.imread("../assets/out/first_object.png")  # read it back
print("saved file exists:", os.path.exists("../assets/out/first_object.png"), "| shape:", saved.shape)

### What you should see / Why we care

A grid of 9 small pictures (each object alone) and `saved file exists: True | shape: (57, 57, 3)`. Open `assets/out/first_object.png` to see the first extracted object.

## Part 2: a messy scene, filtering by shape

`objects_scene.png` has big and small circles, squares, thin bars, triangles, two overlapping circles that merge into one blob, and tiny dust specks. We will remove the wrong things **one filter at a time**.

### What this code does

We load the scene, make it gray, blur it, apply Otsu and flip it so the objects are white. Then we find all contours and define `draw_boxes`, a small recipe that draws a green box around every contour in a list.

In [ ]:
scene = load(SCENE_PATH)                                    # the messy scene
gray = cv2.cvtColor(scene, cv2.COLOR_BGR2GRAY)              # gray
blur = cv2.GaussianBlur(gray, (5, 5), 0)                    # smooth
_, otsu = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
thresh = cv2.bitwise_not(otsu)                              # objects white on black
all_contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

def draw_boxes(picture, contour_list):                      # a recipe named draw_boxes
    result = picture.copy()                                 # draw on a copy
    for contour in contour_list:
        x, y, w, h = cv2.boundingRect(contour)
        cv2.rectangle(result, (x, y), (x + w, y + h), (0, 160, 0), 2)
    return result
print("stage 0 - all contours:", len(all_contours))

### What you should see / Why we care

`stage 0 - all contours: 21` (the 11 real shapes plus 10 specks).

### What this code does

**Filter 1: area.** Anything smaller than 150 pixels is dust.

In [ ]:
stage1 = []                                                 # survivors of filter 1
for contour in all_contours:
    if cv2.contourArea(contour) >= 150:                     # big enough?
        stage1.append(contour)
print("stage 1 - after area filter:", len(stage1), "(removed", len(all_contours) - len(stage1), ")")

### What you should see / Why we care

`stage 1 - after area filter: 11 (removed 10 )`: the dust is gone.

### What this code does

**Filter 2: aspect ratio** = width divided by height. A circle or square is about 1. A long bar is far from 1. We keep shapes between 0.5 and 2.

In [ ]:
stage2 = []                                                 # survivors of filter 2
for contour in stage1:
    x, y, w, h = cv2.boundingRect(contour)
    ratio = w / h                                           # width divided by height
    if 0.5 <= ratio <= 2.0:                                 # roughly "compact"?
        stage2.append(contour)
    else:
        print("removed a bar: ratio =", round(ratio, 2))    # show what we throw away
print("stage 2 - after aspect-ratio filter:", len(stage2))

### What you should see / Why we care

Two lines `removed a bar: ratio = ...` (one very large, one very small) and `stage 2 - after aspect-ratio filter: 9`.

### What this code does

**Filter 3: circularity** = `4 * pi * area / perimeter^2`. A perfect circle gives 1. A square gives about 0.79 and a triangle about 0.6. `cv2.arcLength(contour, True)` measures the perimeter. We keep shapes with circularity of at least 0.85, which means **circles**.

In [ ]:
stage3 = []                                                 # survivors of filter 3
for contour in stage2:
    area = cv2.contourArea(contour)                         # inside size
    perimeter = cv2.arcLength(contour, True)                # length of the outline
    circularity = 4 * np.pi * area / (perimeter * perimeter)   # 1.0 = perfect circle
    print("area", int(area), "-> circularity", round(circularity, 2))
    if circularity >= 0.85:                                 # round enough?
        stage3.append(contour)
print("stage 3 - circles only:", len(stage3))

### What you should see / Why we care

A short table. Circles score about **0.9**, squares **0.79-0.80**, the merged double circle **0.74** and triangles **0.56-0.57**. `stage 3 - circles only: 3`.

### What this code does

We draw all four stages side by side, one subplot per stage, so you can **see** the filtering.

In [ ]:
show_many([draw_boxes(scene, all_contours), draw_boxes(scene, stage1),
           draw_boxes(scene, stage2), draw_boxes(scene, stage3)],
          ["0: all contours", "1: area >= 150", "2: aspect ratio 0.5-2", "3: circularity >= 0.85"],
          columns=2, size=(12, 9))

### What you should see / Why we care

Specks vanish in stage 1, the two bars vanish in stage 2, and in stage 3 only the three clean circles remain. Each filter removed exactly the wrong things.

## Part 3: finding one known object with template matching

Sometimes we do not want "all circles" but **this one specific shape**. We cut a small picture of it (the *template*) and slide it over the scene. At every position OpenCV scores how similar the patch is. The best score marks the best match.

### What this code does

We cut the left triangle out of the scene by slicing (rows 448-523, columns 58-143) and show it. `template.shape` is (height, width).

In [ ]:
template = gray[448:523, 58:143]                            # rows (y) first, then columns (x)
template_h, template_w = template.shape                     # its height and width
print("template size:", template_w, "wide,", template_h, "tall")
show(template, "the template: one triangle", size=(3, 3))

### What you should see / Why we care

`template size: 85 wide, 75 tall` and a picture of a dark triangle.

### What this code does

`cv2.matchTemplate` returns a map of scores (one per position). `cv2.minMaxLoc` finds the lowest and highest score and **where** they are. For `TM_CCOEFF_NORMED` the best match is the **maximum**. `max_loc` is the top-left corner `(x, y)` of the best match.

In [ ]:
result = cv2.matchTemplate(gray, template, cv2.TM_CCOEFF_NORMED)   # score at every position
min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(result)         # best and worst
print("best score:", round(float(max_val), 3), "at (x, y) =", max_loc)
found = scene.copy()                                               # draw on a copy
cv2.rectangle(found, max_loc, (max_loc[0] + template_w, max_loc[1] + template_h), (0, 0, 255), 3)
show(found, "best match (the template came from here, so score = 1.0)", size=(8, 6))

### What you should see / Why we care

`best score: 1.0 at (x, y) = (58, 448)`: it found the place we cut it from (a perfect score). But there is a **second identical triangle** on the right. Where is it?

### What this code does

We keep **every** position with a score of at least 0.97. `np.where` returns the row numbers (`ys`) and column numbers (`xs`) of all positions that pass.

In [ ]:
ys, xs = np.where(result >= 0.97)                           # all good positions
hits = []                                                   # (score, x, y) for each
for i in range(len(xs)):
    hits.append((float(result[ys[i], xs[i]]), int(xs[i]), int(ys[i])))
hits.sort(reverse=True)                                     # best score first
print("raw hits:", len(hits))
print("best five:", hits[:5])

### What you should see / Why we care

About 10 raw hits: several **neighbouring positions** around each triangle (shifting the template by one pixel still looks almost identical). These are duplicates of the same object.

### What this code does

**Non-max suppression** in plain words: *go through the hits from best to worst; keep a hit only if it does not overlap a hit we already kept.* Two boxes overlap when their `x` values differ by less than the template width **and** their `y` values differ by less than its height.

In [ ]:
kept = []                                                   # hits we keep
for score, x, y in hits:                                    # best first
    overlaps = False                                        # assume no overlap
    for kept_x, kept_y in kept:                             # compare with kept hits
        if abs(x - kept_x) < template_w and abs(y - kept_y) < template_h:
            overlaps = True                                 # too close: duplicate
    if not overlaps:
        kept.append((x, y))                                 # a new, separate object
print("objects found:", len(kept), "at", kept)
final = scene.copy()
for x, y in kept:
    cv2.rectangle(final, (x, y), (x + template_w, y + template_h), (0, 0, 255), 3)
show(final, "both identical triangles found, no duplicates", size=(8, 6))

### What you should see / Why we care

`objects found: 2` at two separate positions. The larger triangle on the far right was **not** matched: template matching is **not scale-invariant** (it only finds things of the same size), which is both a limit and a feature.

## Recap
| Task | Tool |
|---|---|
| Pick a colour | `cv2.cvtColor(..., cv2.COLOR_BGR2HSV)` + `cv2.inRange` |
| Keep only that colour | `cv2.bitwise_and(img, img, mask=mask)` |
| Filter shapes | `cv2.contourArea`, `w / h`, `4 * pi * area / perimeter ** 2` |
| Find a known object | `cv2.matchTemplate` + `cv2.minMaxLoc` |
| Remove duplicates | non-max suppression |

**Next:** Notebook 08, the **capstone**: find and read *every digit* in a photo-like picture.